Drude Model Theory: https://www.chem.uci.edu/~lawm/AM%20Ch%201-3.pdf
Plasma constant: http://www.wave-scattering.com/drudefit.html
Real and Imaginary permittivity: https://empossible.net/wp-content/uploads/2020/06/Lecture-Drude-Model-for-Metals.pdf
Free electron density (Cu - Al): https://phys.libretexts.org/Bookshelves/University_Physics/University_Physics_(OpenStax)/University_Physics_III_-_Optics_and_Modern_Physics_(OpenStax)/09%3A_Condensed_Matter_Physics/9.05%3A_Free_Electron_Model_of_Metals


In [37]:
path_root = "/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow"

In [1]:
import pandas as pd
import numpy as np

In [32]:
import numpy as np

def drude_model_alloy_from_rho(
    ne_pure,    # array: electron densities of pure elements [m^-3]
    rho_alloy,  # float: alloy resistivity [ohm*m]
    wvl,        # wavelength (nm)
    comp,       
    eps_inf=1.0,
    m_eff=9.1093837e-31,
):
    eps0 = 8.8541878128e-12  # F/m
    e = 1.602176634e-19      # C (magnitude)
    c = 299792458.0          # m/s

    comp = np.asarray(comp, dtype=float)
    ne_pure = np.asarray(ne_pure, dtype=float)

    # 1) alloy electron density 
    n_alloy = float(np.dot(ne_pure, comp))  # [m^-3]

    # 2) omega from wavelength
    wvl = np.asarray(wvl, dtype=float)
    wvl_m = wvl * 1e-9
    omega = 2 * np.pi * c / wvl_m

    # 3) Drude parameters 
    omega_p2 = n_alloy * e**2 / (eps0 * m_eff)              # ωp^2 [s^-2]
    tau = m_eff / (n_alloy * e**2 * rho_alloy)              # τ [s]

    # 4) dielectric function parts
    denom = 1.0 + (omega * tau)**2
    e1 = eps_inf - (omega_p2 * tau**2) / denom
    e2 = (omega_p2 * tau) / (omega * denom)

    return e1, e2

elem_cols = ["Cu","Al","Ni"]
ne_pure = [8.47e28,18.1e28,5.64e28]
rho_col = 'resistivity'
wvl = 1550

def row_to_result(row):
    comp = row[elem_cols].to_numpy(dtype=float)      # shape (3,)
    rho_alloy = float(row[rho_col])
    return drude_model_alloy_from_rho(
        ne_pure=ne_pure,
        rho_alloy=rho_alloy,
        wvl=wvl,
        comp=comp,
        eps_inf=1.0,
        m_eff=9.1093837e-31,
        wvl_unit="nm",
    )

In [47]:
comp_all_properties = pd.read_csv('cualni_compositions.csv')
resistivity = np.load('electrical_resistivity_predicted.npy') *1e-8 # resistivity intial units (µΩ·cm)

comp_all_properties['resistivity'] = resistivity 

In [48]:
comp_all_properties[["e1", "e2"]] = comp_all_properties.apply(
    lambda row: drude_model_alloy_from_rho(
        ne_pure=ne_pure,
        rho_alloy=float(row[rho_col]),
        wvl=wvl,
        comp=row[elem_cols].to_numpy(dtype=float),
    ),
    axis=1,
    result_type="expand"
)

In [49]:
comp_all_properties.head(5)

,ID,Cu,Ni,Al,r,r_ave,del_r,del_EN,S,VEC,resistivity,e1,e2
0,1,0.0,0.00,1.00,143.00,143.00,0.000000,0.000000,-0.000000,3.00,1.094524e-06,-16.647495,81.068049
1,2,0.0,0.02,0.98,142.62,142.62,0.018651,0.042000,0.815097,2.98,1.089933e-06,-17.014919,81.274201
2,3,0.0,0.04,0.96,142.24,142.24,0.026176,0.058788,1.396288,2.96,1.043620e-06,-18.814453,84.399423
3,4,0.0,0.06,0.94,141.86,141.86,0.031808,0.071246,1.887008,2.94,9.066918e-07,-25.132322,95.336711
4,5,0.0,0.08,0.92,141.48,141.48,0.036433,0.081388,2.317689,2.92,9.360535e-07,-23.935480,92.567537


In [50]:
comp_all_properties.to_pickle(path_root+"/Data_Extraction/Dielectric_calculated.pkl")